In [1]:
# Suppress the Warning of `Windows requires special permissions to create symbolic links (symlinks).`
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Initializing the device
import torch
device = 'cuda' if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
# Input
text = "Dear Amazon, last week I ordered an Optimus Prime action figure from your online store in Germany. Unfortunately, when I opened the package, I discovered to my horror that I had been sent an action figure of Megatron instead! As a lifelong enemy of the Decepticons, I hope you can understand my dilemma. To resolve the issue, I demand an exchange of Megatron for the Optimus Prime figure I ordered. Enclosed are copies of my records concerning this purchase. I expect to hear from you soon. Sincerely, Bumblebee."

In [3]:
# Transformers —→ Pipeline
# Sentement Classification
from transformers import pipeline
# `Pipeline` abstract away all the steps needed to convert raw text into a set of predictions from a fine-tuned model.

classifier = pipeline(
    task="text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    revision="714eb0f",
    device=device
)
classifier

# available tasks —→ 'any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection'

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

TextClassificationPipeline: {'model': 'DistilBertForSequenceClassification', 'dtype': 'float32', 'device': 'cuda', 'input_modalities': 'text'}

In [4]:
# Model used - distilbert-base-uncased-finetuned-sst-2-english
outputs = classifier(text)
outputs  

# score —→ Probability of predicted class

[{'label': 'NEGATIVE', 'score': 0.9015456438064575}]

---

In [5]:
# Named Entity Reconization
from huggingface_hub import snapshot_download
import pathlib

local_path = snapshot_download(
    repo_id="dslim/bert-base-NER",
    local_dir=pathlib.Path() / "Models"
)
# then load from local_path with pipeline(..., model=local_path)

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

In [6]:
ner_tagger = pipeline(
    task="token-classification",
    model=pathlib.Path() / "Models", # Local Model directory Path
    aggregation_strategy="max", # none, simple, first, max
    device=device
)
ner_tagger


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: Models
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TokenClassificationPipeline: {'model': 'BertForTokenClassification', 'dtype': 'float32', 'device': 'cuda', 'input_modalities': 'text'}

In [7]:
import pandas as pd
outputs = ner_tagger(text)
df = pd.DataFrame(outputs)
print(df)

  entity_group     score           word  start  end
0          ORG  0.787448         Amazon      5   11
1         MISC  0.998630  Optimus Prime     36   49
2          LOC  0.999737        Germany     90   97
3          PER  0.926478       Megatron    208  216
4         MISC  0.880146    Decepticons    253  264
5          PER  0.612413       Megatron    350  358
6         MISC  0.998314  Optimus Prime    367  380
7          PER  0.926003      Bumblebee    502  511


---

In [8]:
# Text Generation
from transformers import set_seed
set_seed(42) # Set the seed to get reproducible results

In [ ]:
generator = pipeline(
    task="text-generation",
    # device=device
) # Need more than 8GB RAM

In [ ]:
response = "Dear Bumblebee, I am sorry to hear that your order was mixed up."
prompt = text + "\n\nCustomer service response:\n" + response
outputs = generator(prompt, max_length=200)
print(outputs[0]['generated_text'])